In [ ]:
# ## Import libraries
import random
import time
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.colors import ListedColormap
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, Subset, random_split

In [3]:
# ## Set up variables
DIR_DATA = Path("/content/data")
DIR_TRAIN = DIR_DATA / "train"
DIR_TEST = DIR_DATA / "test"

DIR_OUT_MODEL = Path("5_unet_checkpoints/")

SEED = 42
EPOCHS = 40
AUGMENT = True
IOU_THRESH = 0.70

WETLAND_CMAP = ListedColormap(["white", "tab:blue"])

In [ ]:
# ## Set up classes and functions
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=6, out_channels=1):
        super(UNet, self).__init__()

        # Encoder
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64)

        # Output
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        d1 = self.down1(x)
        p1 = self.pool(d1)

        d2 = self.down2(p1)
        p2 = self.pool(d2)

        d3 = self.down3(p2)
        p3 = self.pool(d3)

        d4 = self.down4(p3)
        p4 = self.pool(d4)

        # Bottleneck
        bn = self.bottleneck(p4)

        # Decoder
        up4 = self.up4(bn)
        up4 = torch.cat([up4, d4], dim=1)
        up4 = self.conv4(up4)

        up3 = self.up3(up4)
        up3 = torch.cat([up3, d3], dim=1)
        up3 = self.conv3(up3)

        up2 = self.up2(up3)
        up2 = torch.cat([up2, d2], dim=1)
        up2 = self.conv2(up2)

        up1 = self.up1(up2)
        up1 = torch.cat([up1, d1], dim=1)
        up1 = self.conv1(up1)

        out = self.out(up1)
        return out

class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)

        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()
        dice = (2. * intersection + self.smooth) / \
               (preds.sum() + targets.sum() + self.smooth)

        return 1 - dice

class WetlandDataset(Dataset):
    def __init__(self, img_dir, mask_dir, augment=False):
        self.img_files = sorted(list(Path(img_dir).glob("*.npy")))
        self.mask_files = sorted(list(Path(mask_dir).glob("*.npy")))
        self.augment = augment

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img = np.load(self.img_files[idx])
        mask = np.load(self.mask_files[idx])

        # Normalize image
        img = img.astype(np.float32) / 10000.0
        mask = mask.astype(np.float32)

        if self.augment:
            img, mask = self.apply_augmentation(img, mask)

        return torch.tensor(img), torch.tensor(mask)

    def apply_augmentation(self, img, mask):
        # Horizontal flip
        if random.random() > 0.5:
            img = np.flip(img, axis=2)
            mask = np.flip(mask, axis=2)

        # Vertical flip
        if random.random() > 0.5:
            img = np.flip(img, axis=1)
            mask = np.flip(mask, axis=1)

        # Rotation 90
        if random.random() > 0.5:
            img = np.rot90(img, k=1, axes=(1, 2))
            mask = np.rot90(mask, k=1, axes=(1, 2))

        return img.copy(), mask.copy()

def iou_score(preds, targets, threshold=0.5):
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection

    return (intersection + 1e-6) / (union + 1e-6)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for imgs, masks in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)
        loss = bce(preds, masks) + dice(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def eval_epoch(model, loader):
    model.eval()
    total_iou = 0
    n = 0

    with torch.no_grad():
        for imgs, masks in tqdm(loader, leave=False):
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            preds = torch.sigmoid(preds)
            preds = (preds > 0.5).float()

            intersection = (preds * masks).sum()
            union = preds.sum() + masks.sum() - intersection

            iou = (intersection + 1e-6) / (union + 1e-6)

            total_iou += iou.item()
            n += 1

    return total_iou / n

def show_prediction(model, dataset, idx):
    model.eval()

    img, mask = dataset[idx]

    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
        pred = torch.sigmoid(pred)
        pred = (pred > 0.5).float()

    img = img.numpy()
    mask = mask.numpy()[0]
    pred = pred.cpu().numpy()[0, 0]

    # Make RGB from bands (B4, B3, B2)
    rgb = img[[2,1,0]]
    rgb = np.transpose(rgb, (1,2,0))
    rgb = rgb / rgb.max()

    # Custom colormap: 0 = white, 1 = blue
    wetland_cmap = ListedColormap(["white", "tab:blue"])

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(rgb)
    plt.title("Image")

    plt.subplot(1,3,2)
    plt.imshow(mask, cmap=WETLAND_CMAP)
    plt.title("Ground Truth")

    plt.subplot(1,3,3)
    plt.imshow(pred, cmap=WETLAND_CMAP)
    plt.title("Prediction")

    plt.show()


In [ ]:
# ## Set up datasets
data_train_full = WetlandDataset(
    DIR_TRAIN / "images",
    DIR_TRAIN / "masks_binary",
    augment=False
)

train_size = int(0.8 * len(data_train_full))
val_size = len(data_train_full) - train_size

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(data_train_full), generator=generator)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

data_train = WetlandDataset(
    DIR_TRAIN / "images",
    DIR_TRAIN / "masks_binary",
    augment=AUGMENT
)

data_val = WetlandDataset(
    DIR_TRAIN / "images",
    DIR_TRAIN / "masks_binary",
    augment=False
)

data_train = Subset(data_train, train_indices)
data_val = Subset(data_val, val_indices)

data_test = WetlandDataset(
    DIR_TEST / "images",
    DIR_TEST / "masks_binary"
)

train_loader = DataLoader(data_train, batch_size=16, shuffle=True)
val_loader = DataLoader(data_val, batch_size=16, shuffle=False)
test_loader = DataLoader(data_test, batch_size=16, shuffle=False)

In [ ]:
# ## Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ## Set up model
model = UNet(in_channels=6, out_channels=1).to(device)

In [ ]:
# ## Set up training parameters
bce = nn.BCEWithLogitsLoss()
dice = DiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# ## Run training
start_time = time.time()

train_losses = []
val_ious = []

best_iou = 0
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(model, train_loader, optimizer)
    val_iou = eval_epoch(model, val_loader)

    train_losses.append(train_loss)
    val_ious.append(val_iou)

    print(f"Train Loss: {train_loss:.4f} | Val IoU: {val_iou:.4f}")
    if (val_iou > best_iou) and (val_iou > IOU_THRESH):
        best_iou = val_iou
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = f"unet_{timestamp}_epoch{str(epoch + 1).zfill(2)}_iou{val_iou:.3f}.pth"

        torch.save(model.state_dict(), DIR_OUT_MODEL / model_name)
        print("Model saved")

end_time = time.time()
print(f"Elapsed time: {end_time - start_time} seconds")

In [ ]:
# ## Load saved model
model = UNet(in_channels=6, out_channels=1)
model.load_state_dict(
    torch.load(
        DIR_OUT_MODEL / "unet_20260328_211654_epoch37_iou0.742.pth"
    )
)
model.to(device)
model.eval()

In [ ]:
test_iou = eval_epoch(model, test_loader)
print("\nFinal Test IoU:", test_iou)

In [ ]:
# 2021 Summer Data; Augmented
# unet_20260328_211654_epoch37_iou0.742.pth
# Final Test IoU: 0.6605874679305337

# 2021 Summer Data; Not Augmented
# unet_20260329_180133_epoch39_iou0.750_noaug.pth
# Final Test IoU: 0.6541184349493547

# 2025 Data; Augmented
# unet_20260321_191231_epoch38_iou0.805.pth
# Final Test IoU: 0.6481862474571575

# 2025 Data; Not Augmented
# unet_epoch30_iou0.807_20260321_200122.pth
# Final Test IoU: 0.6357627375559374

In [ ]:
plt.figure(figsize=(6,4))

epochs = range(1, len(train_losses)+1)

plt.plot(epochs, train_losses, '-o', label="Training Loss")
plt.plot(epochs, val_ious, '-s', label="Validation IoU")

plt.xlabel("Epoch")
plt.title("Training Loss and Validation IoU")
plt.legend()
plt.grid(True)

# Add Test IoU text inside the plot
plt.text(
    0.6 * max(epochs),
    max(train_losses),
    f"Test IoU = {test_iou:.3f}",
    fontsize=10,
    bbox=dict(facecolor='white', alpha=0.7)
)

# Save figure
plt.savefig(
    DIR_OUT_MODEL / f"training_curve_{timestamp}.png",
    dpi=300,
    bbox_inches="tight")

plt.show()

In [ ]:
show_prediction(model, data_test, 0)

# Evaluation

In [ ]:
def evaluate_coverage_per_class(model, dataset, class_values, device):
    model.eval()

    TP = {c: 0 for c in class_values}
    GT = {c: 0 for c in class_values}

    for i in range(len(dataset)):
        img, mask_multi = dataset[i]

        img = img.unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(img)
            pred = torch.sigmoid(pred)
            pred = (pred > 0.5).cpu().numpy()[0, 0]

        mask_multi = mask_multi.numpy()[0]

        for c in class_values:
            gt = (mask_multi == c)

            TP[c] += np.sum(gt & (pred == 1))
            GT[c] += np.sum(gt)

    coverage = {}
    for c in class_values:
        coverage[c] = TP[c] / (GT[c] + 1e-6)

    return coverage

def evaluate_false_positive_rate(model, dataset, device):
    model.eval()

    FP = 0
    NON_WETLAND = 0

    for i in range(len(dataset)):
        img, mask_binary = dataset[i]

        img = img.unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(img)
            pred = torch.sigmoid(pred)
            pred = (pred > 0.5).cpu().numpy()[0, 0]

        mask_binary = mask_binary.numpy()[0]

        non_wetland = (mask_binary == 0)

        FP += np.sum((pred == 1) & non_wetland)
        NON_WETLAND += np.sum(non_wetland)

    return FP / (NON_WETLAND + 1e-6)

def evaluate_band_occlusion(model, test_loader, baseline_iou, device):
    model.eval()

    band_names = ["B2","B3","B4","B8","B11","B12"]
    band_importance = []

    for band in range(6):
        intersections = 0
        unions = 0

        for imgs, masks in test_loader:
            imgs = imgs.clone()
            masks = masks.to(device)

            # Occlude one band
            imgs[:, band, :, :] = 0

            imgs = imgs.to(device)

            with torch.no_grad():
                preds = model(imgs)
                preds = torch.sigmoid(preds)
                preds = (preds > 0.5).float()

            intersection = (preds * masks).sum().item()
            union = ((preds + masks) > 0).sum().item()

            intersections += intersection
            unions += union

        iou = intersections / (unions + 1e-6)
        importance = baseline_iou - iou

        band_importance.append(importance)

        print(f"{band_names[band]} IoU: {iou:.4f}, Importance: {importance:.4f}")

    return band_names, band_importance

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        self.target_layer.register_forward_hook(self.forward_hook)
        self.target_layer.register_backward_hook(self.backward_hook)

    def forward_hook(self, module, input, output):
        self.activations = output

    def backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, input_image):
        self.model.zero_grad()

        output = self.model(input_image)
        output = torch.sigmoid(output)

        # For segmentation, use mean output as target
        loss = output.mean()
        loss.backward()

        gradients = self.gradients
        activations = self.activations

        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * activations, dim=1)

        cam = F.relu(cam)
        cam = cam.squeeze().cpu().detach().numpy()

        cam = cv2.resize(cam, (256, 256))
        cam = (cam - cam.min()) / (cam.max() + 1e-6)

        return cam

def show_gradcam(model, dataset, grad_cam, idx):
    model.eval()

    img, mask = dataset[idx]
    input_img = img.unsqueeze(0).to(device)

    cam = grad_cam.generate(input_img)

    img_np = img.numpy()

    # RGB image from bands
    rgb = img_np[[2,1,0]]
    rgb = np.transpose(rgb, (1,2,0))
    rgb = rgb / rgb.max()

    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(rgb)
    plt.title("Image")

    plt.subplot(1,3,2)
    plt.imshow(mask.numpy()[0], cmap="gray")
    plt.title("Ground Truth")

    plt.subplot(1,3,3)
    plt.imshow(rgb)
    heatmap = plt.imshow(cam, cmap="jet", alpha=0.5)
    plt.title("Grad-CAM")
    plt.colorbar(heatmap, fraction=0.046, pad=0.04)

    plt.show()

In [ ]:
data_test_mc = WetlandDataset(
    DIR_TEST / "images",
    DIR_TEST / "masks_multiclass"
)

test_mc_loader = DataLoader(data_test_mc, batch_size=16, shuffle=False)

In [ ]:
class_values = [1, 2, 3, 4, 5]

coverage = evaluate_coverage_per_class(
    model,
    data_test_mc,
    class_values,
    device
)

print("Coverage per class:")
for c in class_values:
    print(f"Class {c}: {coverage[c]:.3f}")

fpr = evaluate_false_positive_rate(
    model,
    data_test,
    device
)

print("False Positive Rate:", fpr)

In [ ]:
classes = list(coverage.keys())
values = list(coverage.values())

plt.figure(figsize=(6,4))

class_names = {
    1: "Bog",
    2: "Fen",
    3: "Marsh",
    4: "Swamp",
    5: "Water"
}

labels = [class_names[c] for c in classes]

plt.bar(labels, values)
plt.ylabel("Coverage Rate")
plt.title("Coverage Rate per Wetland Type")

for i, v in enumerate(values):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center')

plt.ylim(0, 1)
plt.show()

In [ ]:
band_names, band_importance = evaluate_band_occlusion(
    model,
    test_loader,
    test_iou,
    device
)

In [ ]:
plt.figure(figsize=(6,4))
bars = plt.bar(band_names, band_importance)

plt.xlabel("Spectral Band")
plt.ylabel("IoU Drop")
plt.title("Band Importance (Band Occlusion)")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.002,
        f"{height:.3f}",
        ha='center'
    )

# plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# grad_cam = GradCAM(model, model.down1)
# grad_cam = GradCAM(model, model.bottleneck)
grad_cam = GradCAM(model, model.conv1)
show_gradcam(model, data_test, grad_cam, 48)